# Week 1: behavior run on Gemma-2-2B
Runtime: **T4 GPU**, High-RAM recommended. Secrets (key icon): `HF_TOKEN`, and `GITHUB_TOKEN` if you clone the private repo. Run the cells top to bottom.

## 1. Get the code
Either clone with a GitHub token (a fine-grained token with read-only **Contents** access to this repo), or upload a zip of the repo in the file panel and unzip it. The cell skips cloning if the folder already exists.

In [ ]:
import os, subprocess
os.chdir("/content")
if not os.path.exists("textbook-vs-data"):
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/acrodrig98/textbook-vs-data.git"], check=True)
    # keep the token out of the clone's saved settings
    subprocess.run(["git", "-C", "textbook-vs-data", "remote", "set-url", "origin",
                    "https://github.com/acrodrig98/textbook-vs-data.git"], check=True)
%cd /content/textbook-vs-data
!git log --oneline -3

## 2. Install and log in
Only what the behavior run needs; circuit-tracer comes in Week 2. If Colab asks to restart the session after the install, do it, then run this cell again. Every cell below moves into the repo folder first, so the restart is harmless.

In [ ]:
%cd /content/textbook-vs-data
!pip install -q transformers==4.57.3 huggingface_hub==0.36.2 pytest
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

## 3. Tests and prompts

In [ ]:
%cd /content/textbook-vs-data
!python -m pytest -q tests/test_generate.py tests/test_eval.py
!python -m prompts.generate --out data/week1.jsonl --preview 1

## 4. Quick check, then the full run
The quick check scores 64 prompts. If it looks fine, the full run takes several minutes.

In [ ]:
%cd /content/textbook-vs-data
!HF_HUB_DISABLE_PROGRESS_BARS=1 python -m eval.behavior --model google/gemma-2-2b --limit 64 --out results/quickcheck.jsonl

In [ ]:
%cd /content/textbook-vs-data
!HF_HUB_DISABLE_PROGRESS_BARS=1 python -m eval.behavior --model google/gemma-2-2b --prompts data/week1.jsonl

## 5. Phase diagrams and go/no-go

In [ ]:
%cd /content/textbook-vs-data
!python -m eval.analyze results/week1_gemma-2-2b.jsonl
from IPython.display import Image, display
display(Image("figures/week1_gemma-2-2b_phase.png"))
display(Image("figures/week1_gemma-2-2b_levels.png"))

## 6. Download the results
Commit these to the repo from your Mac.

In [ ]:
%cd /content/textbook-vs-data
!zip -q -r week1_results.zip results/week1_gemma-2-2b* figures/week1_gemma-2-2b*
from google.colab import files
files.download("week1_results.zip")

## 7. Control: made-up law names (Qwen3-4B)
Same tables and wording as the law-named prompts, except the law's name is invented ("A Varnek's law experiment ..."). Only the new prompts are scored; they line up with the earlier Qwen results already in `results/` (same tables), and the analysis merges the two. Takes about 5 minutes on a T4.

In [ ]:
%cd /content/textbook-vs-data
!python -m prompts.generate --out data/week1_fakelaw.jsonl --levels fake_named --formats mc --preview 1
!HF_HUB_DISABLE_PROGRESS_BARS=1 python -m eval.behavior --model Qwen/Qwen3-4B --prompts data/week1_fakelaw.jsonl

In [ ]:
%cd /content/textbook-vs-data
!python -m eval.analyze results/week1_Qwen3-4B.jsonl results/week1_fakelaw_Qwen3-4B.jsonl --name week1_control_Qwen3-4B
from IPython.display import Image, display
display(Image("figures/week1_control_Qwen3-4B_levels.png"))
display(Image("figures/week1_control_Qwen3-4B_phase.png"))

In [ ]:
%cd /content/textbook-vs-data
!zip -q -r week1_control.zip results/week1_fakelaw_Qwen3-4B* results/week1_control_Qwen3-4B* figures/week1_control_Qwen3-4B*
from google.colab import files
files.download("week1_control.zip")

## 8. Control: setup only, no law name (Qwen3-4B)
The law-named prompts with only the law's name removed; the description of the setup ("... for a resistor", "a dropped ball falls ...") is kept word for word. Merged with the earlier Qwen and made-up-law results, this splits the pull into: the law's name, generic "X's law" framing, the setup description, and the variable names. About 5 minutes on a T4. Run sections 1-2 first in a fresh session; you can skip 3-7.

In [ ]:
%cd /content/textbook-vs-data
!python -m prompts.generate --out data/week1_setting.jsonl --levels setting_only --formats mc --preview 1
!HF_HUB_DISABLE_PROGRESS_BARS=1 python -m eval.behavior --model Qwen/Qwen3-4B --prompts data/week1_setting.jsonl

In [ ]:
%cd /content/textbook-vs-data
!python -m eval.analyze results/week1_Qwen3-4B.jsonl results/week1_fakelaw_Qwen3-4B.jsonl results/week1_setting_Qwen3-4B.jsonl --name week1_breakdown_Qwen3-4B
from IPython.display import Image, display
display(Image("figures/week1_breakdown_Qwen3-4B_levels.png"))
display(Image("figures/week1_breakdown_Qwen3-4B_phase.png"))

In [ ]:
%cd /content/textbook-vs-data
!zip -q -r week1_breakdown.zip results/week1_setting_Qwen3-4B* results/week1_breakdown_Qwen3-4B* figures/week1_breakdown_Qwen3-4B*
from google.colab import files
files.download("week1_breakdown.zip")